# 🤖 Hierarchical Dual-Frequency DRL Trading Bot

**Architecture**: Macro LLM Swarm (news → 5 LLM personas → consensus scalars + embedding,
**time-aligned** to LOB rows) + Micro CVML (10-level LOB → Conv2D → 64D) → PPO.

**Protocol**: models train on the first 70% of the data (random episode starts) and are
evaluated **out-of-sample** on the held-out last 30%, against Buy&Hold / TWAP / VWAP / Random
baselines and two ablations (macro zeroed, flat MLP).

Smoke-test config: **10,000 timesteps** per model.
This notebook is generated by `build_notebook.py` — module code is inlined from the repo
sources automatically. Do not edit the inlined cells by hand.

---

In [ ]:
# ── Install Dependencies ─────────────────────────────────────────────────────
!pip install -q stable-baselines3 gymnasium

import os, sys, time, csv
import numpy as np
import pandas as pd
import torch
import gymnasium as gym
from gymnasium import spaces
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# ── CONFIGURATION ────────────────────────────────────────────────────────────
KAGGLE_INPUT = "/kaggle/input/drl-trading-lob-data"
ON_KAGGLE = os.path.exists(KAGGLE_INPUT)

DATA_DIR = KAGGLE_INPUT if ON_KAGGLE else "data/raw"
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "notebooks/out"
MODELS_DIR = os.path.join(OUT_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

DATA_PATH = os.path.join(DATA_DIR, "lobster_aapl_10_level.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join(DATA_DIR, "synthetic_lob_10_level.csv")

MACRO_PATH = os.path.join(DATA_DIR, "macro_vectors.npz")
if not os.path.exists(MACRO_PATH):
    MACRO_PATH = None

TRAIN_TIMESTEPS = 10000   # smoke test; set 500_000 for a full run
REWARD_TYPE = "log_return"            # "log_return" | "sharpe" | "pnl"
TRAIN_FRAC = 0.7                      # first 70% of rows = training window
TRAIN_EPISODE_STEPS = 256             # episode length during training

RUN_LOG = os.path.join(OUT_DIR, "run_log.txt")
def log_line(msg):
    print(msg)
    with open(RUN_LOG, "a") as f:
        f.write(msg + "\n")

log_line(f"data={DATA_PATH}")
log_line(f"macro={MACRO_PATH}")
log_line(f"timesteps={TRAIN_TIMESTEPS:,} reward={REWARD_TYPE} device={DEVICE}")

## 1. CVML — Convolutional Cross-Variate Mixing Layer
Depthwise conv over adjacent price levels + pointwise conv mixing the 4 variates.
Input layout must match `canonical_lob_columns()`: `[bid_p1..10, ask_p1..10, bid_s1..10, ask_s1..10]`.

In [ ]:
# ── Inlined from micro/cvml.py (auto-generated by build_notebook.py) ──
import torch
import torch.nn as nn

class CVML(nn.Module):
    """
    Convolutional Cross-Variate Mixing Layer (CVML) for Limit Order Book data.
    Takes flat (40,) features representing 10 levels of (bid_p, ask_p, bid_s, ask_s)
    and reconstructs them into 2D spatial features for depthwise and pointwise convolution.
    """
    def __init__(self, in_channels=4, levels=10, out_dim=64):
        super(CVML, self).__init__()
        self.levels = levels
        self.in_channels = in_channels
        
        # Depthwise convolution along the price levels (groups=in_channels)
        self.depthwise = nn.Conv2d(
            in_channels=self.in_channels,
            out_channels=self.in_channels,
            kernel_size=(3, 1), # convolve along 3 adjacent levels
            padding=(1, 0),
            groups=self.in_channels
        )
        
        # Pointwise convolution to mix the 4 variates (bid price, bid size, ask price, ask size)
        self.pointwise = nn.Conv2d(
            in_channels=self.in_channels,
            out_channels=16,
            kernel_size=1
        )
        
        self.activation = nn.ReLU()
        self.flatten = nn.Flatten()
        
        # Output dimension: 16 channels * 10 levels * 1 width = 160 flat features
        # Project down to out_dim (64)
        self.projection = nn.Linear(16 * self.levels, out_dim)
        
    def forward(self, x):
        # x input shape is (Batch, 40)
        batch_size = x.shape[0]
        
        # Note: Depending on the specific dataset ordering, you must reshape carefully.
        # Assuming the 40 features are: 
        # [bid_p1..p10, ask_p1..p10, bid_s1..s10, ask_s1..s10]
        # Reshape to (Batch, Channels=4, Levels=10, Width=1)
        x_reshaped = x.view(batch_size, self.in_channels, self.levels, 1)
        
        # Apply depthwise
        out = self.depthwise(x_reshaped)
        out = self.activation(out)
        
        # Apply pointwise
        out = self.pointwise(out)
        out = self.activation(out)
        
        # Flatten and project
        out = self.flatten(out)
        out = self.projection(out)
        
        return out


## 2. LOBReplayEnv — Gymnasium Environment
Windowed replay (train/test split), random episode starts, market frictions,
and **time-aligned macro vectors**: each row sees the most recent news event at
or before its own timestamp (4 consensus scalars + 128 pooled embedding dims).

In [ ]:
# ── Inlined from micro/env.py (auto-generated by build_notebook.py) ──
"""
LOBReplayEnv — Custom Gymnasium environment for LOB replay.

Supports:
- Real LOBSTER data and synthetic data (auto-detection)
- Configurable reward functions: log-return, sharpe, pnl
- Market frictions: taker fees, probabilistic maker rebates
- Time-aligned pre-computed macro vectors from the LLM swarm
- Train/test windowing (start_frac/end_frac) and random episode starts
- Inventory penalty to discourage excessive accumulation
"""

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import os

# Macro observation layout: 4 consensus scalars + 128 pooled embedding dims
MACRO_SCALAR_DIM = 4
MACRO_EMBED_DIM = 128
MACRO_DIM = MACRO_SCALAR_DIM + MACRO_EMBED_DIM  # 132


def canonical_lob_columns(levels=10):
    """Canonical 40-column order shared with CVML's (4, levels) reshape:
    [bid_p1..N, ask_p1..N, bid_s1..N, ask_s1..N] — numeric level order."""
    rng = range(1, levels + 1)
    return (
        [f"bid_price_{i}" for i in rng]
        + [f"ask_price_{i}" for i in rng]
        + [f"bid_size_{i}" for i in rng]
        + [f"ask_size_{i}" for i in rng]
    )


class LOBReplayEnv(gym.Env):
    """
    A custom Gymnasium replay environment that iterates over historical LOB data.

    Args:
        data_path: Path to LOB CSV (40 price/size columns required)
        use_macro_vector: Whether to include the 132D macro vector in obs
        macro_vectors_path: Path to .npz with keys: timestamps (N,),
            scalars (N,4: direction, magnitude, confidence, agreement),
            embeddings (N,384). Vectors are aligned to LOB rows by timestamp:
            each row sees the most recent event at or before its own time.
        zero_macro: Ablation switch — keep the obs shape but feed zeros
        reward_type: 'log_return' | 'sharpe' | 'pnl' (default: 'log_return')
        inventory_penalty: Quadratic penalty coefficient for large positions
        inactivity_penalty: Per-step cost for consecutive holds (default 0 = off)
        initial_cash: Starting cash — also serves as the natural position limit
        max_steps: Max steps per episode (None = run to end of data window)
        start_frac / end_frac: Fraction of the dataset this env may replay
            (e.g. train 0.0–0.7, eval 0.7–1.0)
        random_start: Randomize the episode start row within the window
    """

    REWARD_TYPES = ("log_return", "sharpe", "pnl")

    def __init__(
        self,
        data_path,
        window_size=50,
        use_macro_vector=True,
        macro_vectors_path=None,
        zero_macro=False,
        reward_type="log_return",
        inventory_penalty=0.0001,
        inactivity_penalty=0.0,
        initial_cash=10000.0,
        max_steps=None,
        start_frac=0.0,
        end_frac=1.0,
        random_start=False,
    ):
        super().__init__()

        self.data_path = data_path
        self.window_size = window_size
        self.use_macro_vector = use_macro_vector
        self.zero_macro = zero_macro
        self.reward_type = reward_type
        self.inventory_penalty = inventory_penalty
        self.inactivity_penalty = inactivity_penalty
        self.initial_cash = initial_cash
        self.random_start = random_start

        assert reward_type in self.REWARD_TYPES, f"reward_type must be one of {self.REWARD_TYPES}"
        assert 0.0 <= start_frac < end_frac <= 1.0, "need 0 <= start_frac < end_frac <= 1"

        # ── Load Data ─────────────────────────────────────────────────────
        self.df = pd.read_csv(data_path)

        # Canonical order — must match CVML's (Batch, 4, levels, 1) reshape.
        self.feature_cols = canonical_lob_columns(levels=10)
        missing = [c for c in self.feature_cols if c not in self.df.columns]
        assert not missing, f"LOB data missing columns: {missing[:4]}..."

        # ── Normalize prices for stable training ──────────────────────────
        self._mid_prices = (
            (self.df["bid_price_1"] + self.df["ask_price_1"]) / 2.0
        ).values
        self._price_scale = self._mid_prices.mean()

        # ── Data window (train/test split) ────────────────────────────────
        n = len(self.df)
        self._window_lo = int(n * start_frac)
        self._window_hi = int(n * end_frac) - 1  # last replayable row index
        assert self._window_hi - self._window_lo >= 2, "data window too small"
        self.max_steps = max_steps  # per-episode step cap (None = to window end)
        self.max_step = self._window_hi  # kept for baseline scripts
        self.current_step = self._window_lo
        self._episode_steps = 0

        # ── Row timestamps as epoch seconds (numeric or datetime strings) ──
        self._row_ts = None
        if "timestamp" in self.df.columns:
            ts_col = self.df["timestamp"]
            if pd.api.types.is_numeric_dtype(ts_col):
                self._row_ts = ts_col.to_numpy(dtype=np.float64)
            else:
                try:
                    dt = pd.to_datetime(ts_col, utc=True)
                    # Unit-independent epoch seconds (ns vs us resolution)
                    self._row_ts = (
                        (dt - pd.Timestamp("1970-01-01", tz="UTC")) / pd.Timedelta(seconds=1)
                    ).to_numpy(dtype=np.float64)
                except (ValueError, TypeError):
                    self._row_ts = None

        # Median bar spacing (for Sharpe annualization)
        if self._row_ts is not None and len(self._row_ts) > 1:
            self._bar_seconds = float(np.median(np.diff(self._row_ts)))
        else:
            self._bar_seconds = 1.0

        # ── Time-aligned macro vectors ────────────────────────────────────
        # Pre-computed per-row: row i sees the latest news event published
        # at or before its timestamp. Rows before the first event see zeros.
        self._macro_by_row = None
        if use_macro_vector and macro_vectors_path and os.path.exists(macro_vectors_path):
            self._macro_by_row = self._build_aligned_macro(macro_vectors_path)

        # ── Action & Observation Spaces ───────────────────────────────────
        # Actions: 0=Hold, 1=Market Buy, 2=Market Sell, 3=Limit Buy, 4=Limit Sell
        self.action_space = spaces.Discrete(5)

        self.lob_space = spaces.Box(low=-10, high=10, shape=(40,), dtype=np.float32)

        if self.use_macro_vector:
            self.macro_space = spaces.Box(low=-1.0, high=1.0, shape=(MACRO_DIM,), dtype=np.float32)
            self.observation_space = spaces.Dict({
                "lob": self.lob_space,
                "macro": self.macro_space,
            })
        else:
            self.observation_space = self.lob_space

        # ── Portfolio State ───────────────────────────────────────────────
        self.inventory = 0
        self.cash = self.initial_cash
        self.prev_portfolio_value = self.initial_cash

        # Rolling reward buffer for Sharpe ratio calculation
        self._reward_buffer = []
        self._sharpe_window = 50

        # Trade tracking
        self.trades = []
        self.portfolio_history = []
        self._zero_macro_vec = np.zeros(MACRO_DIM, dtype=np.float32)

    def _build_aligned_macro(self, npz_path):
        """Load {timestamps, scalars, embeddings} and align to LOB rows."""
        data = np.load(npz_path, allow_pickle=True)
        if not all(k in data for k in ("timestamps", "scalars", "embeddings")):
            print(f"[ENV] {os.path.basename(npz_path)} lacks timestamps/scalars/embeddings "
                  f"(old format?) — macro obs will be zeros")
            return None
        if self._row_ts is None:
            print("[ENV] LOB data has no usable timestamp column — macro obs will be zeros")
            return None

        ts = data["timestamps"].astype(np.float64)
        scalars = data["scalars"].astype(np.float32)          # (N, 4)
        emb = data["embeddings"].astype(np.float32)           # (N, 384)

        order = np.argsort(ts)
        ts, scalars, emb = ts[order], scalars[order], emb[order]

        # Pool 384 → 128 (mean over consecutive triples), then L2-normalize
        pooled = emb.reshape(len(emb), MACRO_EMBED_DIM, -1).mean(axis=2)
        norms = np.maximum(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-8)
        pooled = pooled / norms

        vectors = np.concatenate([np.clip(scalars, -1.0, 1.0), pooled], axis=1)  # (N, 132)

        row_ts = self._row_ts
        idx = np.searchsorted(ts, row_ts, side="right") - 1   # latest event ≤ row time
        aligned = np.zeros((len(self.df), MACRO_DIM), dtype=np.float32)
        has_event = idx >= 0
        aligned[has_event] = vectors[idx[has_event]]

        print(f"[ENV] Aligned {len(ts)} macro events to {int(has_event.sum())}/{len(self.df)} LOB rows")
        return aligned

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)

        lo, hi = self._window_lo, self._window_hi
        if self.random_start:
            # Leave room for at least a short episode
            span = self.max_steps if self.max_steps else (hi - lo)
            latest = max(lo, hi - max(2, min(span, hi - lo)))
            self.current_step = int(self.np_random.integers(lo, latest + 1))
        else:
            self.current_step = lo
        self._episode_steps = 0

        self.inventory = 0
        self.cash = self.initial_cash
        self.prev_portfolio_value = self.initial_cash
        self._reward_buffer = []
        self.trades = []
        self.portfolio_history = [self.initial_cash]
        self._consecutive_holds = 0

        return self._get_obs(), {}

    def _get_macro_obs(self):
        if self.zero_macro or self._macro_by_row is None:
            return self._zero_macro_vec
        return self._macro_by_row[self.current_step]

    def _get_obs(self):
        row = self.df.iloc[self.current_step]
        lob_features = row[self.feature_cols].values.astype(np.float32)

        # Normalize features: prices relative to mid, sizes log-scaled
        # Prices: (price - mid) / mid  → centered around 0 (basis points)
        # Sizes: log1p(size) / 10     → compressed scale
        mid = self._mid_prices[self.current_step]
        if mid > 0:
            # First 20 cols are prices, next 20 are sizes (canonical order)
            lob_features[:20] = (lob_features[:20] - mid) / mid * 100
            lob_features[20:] = np.log1p(lob_features[20:]) / 10.0

        if self.use_macro_vector:
            return {
                "lob": lob_features,
                "macro": self._get_macro_obs(),
            }
        return lob_features

    def _get_mid_price(self):
        row = self.df.iloc[self.current_step]
        return (float(row["bid_price_1"]) + float(row["ask_price_1"])) / 2.0

    def _get_portfolio_value(self):
        mid = self._get_mid_price()
        return self.cash + (self.inventory * mid)

    def step(self, action):
        row = self.df.iloc[self.current_step]
        best_bid = float(row["bid_price_1"])
        best_ask = float(row["ask_price_1"])
        mid_price = (best_bid + best_ask) / 2.0

        # ── Friction Model ────────────────────────────────────────────────
        maker_fee = -0.0001   # rebate (1 basis point)
        taker_fee = 0.0003    # cost (3 basis points)

        trade_executed = False
        execution_price = 0.0
        trade_side = None

        if action == 1:  # Market Buy (cross the spread, take from asks)
            execution_price = best_ask * (1 + taker_fee)
            if self.cash >= execution_price:  # Dynamic cash check
                self.cash -= execution_price
                self.inventory += 1
                trade_executed = True
                trade_side = "buy"

        elif action == 2:  # Market Sell (cross the spread, take from bids)
            # Allow shorting up to initial cash value margin
            execution_price = best_bid * (1 - taker_fee)
            if self.inventory > -(self.initial_cash / max(mid_price, 1e-8)):
                self.cash += execution_price
                self.inventory -= 1
                trade_executed = True
                trade_side = "sell"

        elif action == 3:  # Limit Buy at Best Bid
            fill_prob = min(0.5, float(row.get("bid_size_1", 500)) / 1000.0)
            execution_price = best_bid * (1 + maker_fee)
            if self.np_random.random() < fill_prob and self.cash >= execution_price:
                self.cash -= execution_price
                self.inventory += 1
                trade_executed = True
                trade_side = "limit_buy"

        elif action == 4:  # Limit Sell at Best Ask
            fill_prob = min(0.5, float(row.get("ask_size_1", 500)) / 1000.0)
            execution_price = best_ask * (1 - maker_fee)
            if self.np_random.random() < fill_prob and self.inventory > -(self.initial_cash / max(mid_price, 1e-8)):
                self.cash += execution_price
                self.inventory -= 1
                trade_executed = True
                trade_side = "limit_sell"

        # Record trade
        if trade_executed:
            self.trades.append({
                "step": self.current_step,
                "action": int(action),
                "side": trade_side,
                "price": execution_price,
                "inventory": self.inventory,
                "mid_price": mid_price,
            })

        # ── Advance time ──────────────────────────────────────────────────
        self.current_step += 1
        self._episode_steps += 1
        done = self.current_step >= self._window_hi
        if self.max_steps is not None and self._episode_steps >= self.max_steps:
            done = True
        truncated = False

        # ── Compute Reward ────────────────────────────────────────────────
        portfolio_value = self._get_portfolio_value()
        self.portfolio_history.append(portfolio_value)

        reward = self._compute_reward(portfolio_value)

        # Quadratic inventory penalty — small positions are cheap,
        # large positions get increasingly expensive
        inv_penalty = self.inventory_penalty * (self.inventory ** 2)
        reward -= inv_penalty

        # Optional inactivity penalty (off by default — fees and log-return
        # should shape behavior, not a bribe to trade)
        if action == 0:
            self._consecutive_holds += 1
            if self.inactivity_penalty > 0:
                reward -= self.inactivity_penalty * min(self._consecutive_holds / 10.0, 1.0)
        else:
            self._consecutive_holds = 0

        self.prev_portfolio_value = portfolio_value

        info = {
            "portfolio_value": portfolio_value,
            "inventory": self.inventory,
            "mid_price": mid_price,
            "trade_executed": trade_executed,
            "step": self.current_step,
        }

        return self._get_obs(), float(reward), done, truncated, info

    def _compute_reward(self, portfolio_value):
        """Compute reward based on the configured reward type."""

        if self.reward_type == "log_return":
            # Log-return: log(PV_t / PV_{t-1}), guarded against negative/zero PV
            pv = max(portfolio_value, 1e-8)
            prev_pv = max(self.prev_portfolio_value, 1e-8)
            log_ret = np.log(pv / prev_pv)
            return float(np.clip(log_ret * 10000, -100, 100))  # Clip to prevent extreme rewards

        elif self.reward_type == "sharpe":
            # Rolling Sharpe ratio
            if self.prev_portfolio_value > 0:
                ret = (portfolio_value - self.prev_portfolio_value) / self.prev_portfolio_value
                self._reward_buffer.append(ret)

                if len(self._reward_buffer) >= self._sharpe_window:
                    window = self._reward_buffer[-self._sharpe_window:]
                    mean_ret = np.mean(window)
                    std_ret = np.std(window)
                    sharpe = mean_ret / (std_ret + 1e-8)
                    return float(sharpe)
                else:
                    return float(ret * 100)
            return 0.0

        elif self.reward_type == "pnl":
            # Simple PnL delta (original method, improved)
            pnl_delta = portfolio_value - self.prev_portfolio_value
            return float(pnl_delta / self.initial_cash * 100)

        return 0.0

    def get_episode_stats(self):
        """Compute summary statistics for the completed episode."""
        pv = np.array(self.portfolio_history)
        returns = np.diff(pv) / pv[:-1] if len(pv) > 1 else np.array([0.0])

        total_return = (pv[-1] - pv[0]) / pv[0] if pv[0] > 0 else 0.0
        # Annualize by actual bar spacing: 252 trading days × 6.5h
        periods_per_year = (252 * 6.5 * 3600) / max(self._bar_seconds, 1e-8)
        sharpe = (np.mean(returns) / (np.std(returns) + 1e-8)) * np.sqrt(periods_per_year) if len(returns) > 1 else 0.0
        max_drawdown = np.min(pv / np.maximum.accumulate(pv) - 1) if len(pv) > 1 else 0.0

        buy_trades = [t for t in self.trades if t["side"] in ("buy", "limit_buy")]
        sell_trades = [t for t in self.trades if t["side"] in ("sell", "limit_sell")]

        return {
            "total_return_pct": round(total_return * 100, 4),
            "sharpe_ratio": round(float(sharpe), 4),
            "max_drawdown_pct": round(float(max_drawdown) * 100, 4),
            "total_trades": len(self.trades),
            "buy_trades": len(buy_trades),
            "sell_trades": len(sell_trades),
            "final_pv": round(float(pv[-1]), 2),
            "final_inventory": self.inventory,
        }


## 3. Feature Extractors — SB3 Custom Policies

In [ ]:
# ── Inlined from micro/policy.py (auto-generated by build_notebook.py) ──
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from gymnasium import spaces


class HierarchicalFeatureExtractor(BaseFeaturesExtractor):
    """
    Custom Feature Extractor for Stable Baselines 3 PPO.
    Takes a Dict observation space with:
    - lob: (40,) flat features
    - macro: (macro_dim,) consensus scalars + pooled semantic embedding
    Passes LOB through CVML, then concatenates with Macro.
    """
    def __init__(self, observation_space: spaces.Dict, cvml_out_dim: int = 64, cnn_activation=nn.ReLU):
        macro_dim = observation_space["macro"].shape[0]
        super(HierarchicalFeatureExtractor, self).__init__(observation_space, features_dim=cvml_out_dim + macro_dim)

        self.cvml = CVML(in_channels=4, levels=10, out_dim=cvml_out_dim)
        
        # Optional: Projection layer to avoid macro vector dominating the CVML features
        # For now, we will just pass the macro vector directly into the concatenation
        
    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        # Stable Baselines 3 passes the Dict observation as a dictionary of tensors
        lob_data = observations["lob"]
        macro_data = observations["macro"]
        
        # Ensure correct dtype
        lob_data = lob_data.float()
        macro_data = macro_data.float()
        
        # 1. Process LOB through the Convolutional Cross-Variate Mixing Layer
        cvml_features = self.cvml(lob_data)
        
        # 2. Concatenate the temporal microstructure features with the semantic macro features
        # Shape: (Batch, cvml_out_dim + macro_dim)
        combined_features = torch.cat([cvml_features, macro_data], dim=1)
        
        return combined_features

# For pure LOB testing (ablation study)
class FlatMLPFeatureExtractor(BaseFeaturesExtractor):
    """Ablation baseline feature extractor using flat MLPs instead of CVML."""
    def __init__(self, observation_space: spaces.Dict, mlp_out_dim: int = 64):
        macro_dim = observation_space["macro"].shape[0]
        super(FlatMLPFeatureExtractor, self).__init__(observation_space, features_dim=mlp_out_dim + macro_dim)
        self.mlp = nn.Sequential(
            nn.Linear(40, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )
    
    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        lob_data = observations["lob"].float()
        macro_data = observations["macro"].float()
        mlp_features = self.mlp(lob_data)
        return torch.cat([mlp_features, macro_data], dim=1)

## 4. Training — PPO variants
- **PPO+CVML+Macro** — full model
- **PPO+CVML (no macro)** — macro vector zeroed (macro ablation)
- **PPO+FlatMLP** — flat MLP instead of CVML (architecture ablation)

In [ ]:
HYPERPARAMS = {
    "learning_rate": 3e-4,
    "n_steps": 2048,
    "batch_size": 64,
    "n_epochs": 10,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "max_grad_norm": 0.5,
    "clip_range": 0.2,
}


def make_env(train=True, zero_macro=False, max_steps=None):
    if train:
        return LOBReplayEnv(
            data_path=DATA_PATH, use_macro_vector=True, macro_vectors_path=MACRO_PATH,
            zero_macro=zero_macro, reward_type=REWARD_TYPE, inventory_penalty=0.001,
            start_frac=0.0, end_frac=TRAIN_FRAC, random_start=True,
            max_steps=TRAIN_EPISODE_STEPS)
    return LOBReplayEnv(
        data_path=DATA_PATH, use_macro_vector=True, macro_vectors_path=MACRO_PATH,
        zero_macro=zero_macro, reward_type=REWARD_TYPE, inventory_penalty=0.001,
        start_frac=TRAIN_FRAC, end_frac=1.0, random_start=False, max_steps=max_steps)


def linear_schedule(initial_lr):
    def schedule(progress_remaining):
        return progress_remaining * initial_lr
    return schedule


class MetricsLoggerCallback(BaseCallback):
    def __init__(self, log_path, verbose=0):
        super().__init__(verbose)
        self.log_path = log_path
        self.episode_rewards, self.episode_lengths = [], []
        with open(self.log_path, "w", newline="") as f:
            csv.writer(f).writerow([
                "timestep", "episodes", "mean_reward", "mean_length",
                "entropy_loss", "policy_loss", "value_loss", "learning_rate"])

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])
        return True

    def _on_rollout_end(self):
        if not self.episode_rewards:
            return
        logger = self.model.logger
        with open(self.log_path, "a", newline="") as f:
            csv.writer(f).writerow([
                self.num_timesteps, len(self.episode_rewards),
                np.mean(self.episode_rewards[-10:]), np.mean(self.episode_lengths[-10:]),
                logger.name_to_value.get("train/entropy_loss", 0),
                logger.name_to_value.get("train/policy_gradient_loss", 0),
                logger.name_to_value.get("train/value_loss", 0),
                self.model.lr_schedule(self.model._current_progress_remaining)])


def train_model(model_name, use_cvml=True, zero_macro=False,
                total_timesteps=TRAIN_TIMESTEPS):
    log_line(f"\n=== TRAIN {model_name} | {total_timesteps:,} steps ===")
    env = make_env(train=True, zero_macro=zero_macro)
    extractor = HierarchicalFeatureExtractor if use_cvml else FlatMLPFeatureExtractor
    model = PPO(
        "MultiInputPolicy", env,
        policy_kwargs={"features_extractor_class": extractor,
                       "features_extractor_kwargs": {},
                       "net_arch": [256, 128, 64]},
        verbose=1,
        learning_rate=linear_schedule(HYPERPARAMS["learning_rate"]),
        n_steps=HYPERPARAMS["n_steps"], batch_size=HYPERPARAMS["batch_size"],
        n_epochs=HYPERPARAMS["n_epochs"], gamma=HYPERPARAMS["gamma"],
        gae_lambda=HYPERPARAMS["gae_lambda"], ent_coef=HYPERPARAMS["ent_coef"],
        vf_coef=HYPERPARAMS["vf_coef"], max_grad_norm=HYPERPARAMS["max_grad_norm"],
        clip_range=HYPERPARAMS["clip_range"], device=DEVICE)

    t0 = time.time()
    model.learn(total_timesteps=total_timesteps,
                callback=[MetricsLoggerCallback(os.path.join(OUT_DIR, f"training_log_{model_name}.csv"))])
    model.save(os.path.join(MODELS_DIR, f"{model_name}_final"))
    log_line(f"{model_name}: trained in {time.time()-t0:.1f}s, saved to {MODELS_DIR}")
    return model

### 4a. Train all three variants

In [ ]:
cvml_model = train_model("ppo_cvml", use_cvml=True)

In [ ]:
nomacro_model = train_model("ppo_cvml_nomacro", use_cvml=True, zero_macro=True)

In [ ]:
flat_model = train_model("ppo_flat", use_cvml=False)

## 5. Out-of-Sample Evaluation — held-out last 30% of the data

In [ ]:
def evaluate_agent(model, env, name="Agent"):
    obs, _ = env.reset()
    done, total_reward = False, 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(action)
        total_reward += reward
        if truncated:
            break
    stats = env.get_episode_stats()
    stats["total_reward"] = round(total_reward, 4)
    stats["name"] = name
    return stats


def run_buy_hold(env):
    obs, _ = env.reset()
    done = False
    while not done:
        row = env.df.iloc[env.current_step]
        action = 1 if env.cash >= float(row["ask_price_1"]) * 1.001 else 0
        obs, r, done, trunc, info = env.step(action)
        if trunc:
            break
    stats = env.get_episode_stats(); stats["name"] = "Buy&Hold"
    return stats


def run_twap(env):
    obs, _ = env.reset()
    done, steps = False, 0
    horizon = env._window_hi - env.current_step
    while not done:
        if steps % 50 == 0 and steps < horizon - 100:
            action = 1
        elif steps >= horizon - 50 and env.inventory > 0:
            action = 2
        else:
            action = 0
        obs, r, done, trunc, info = env.step(action)
        steps += 1
        if trunc:
            break
    stats = env.get_episode_stats(); stats["name"] = "TWAP"
    return stats


def run_vwap(env):
    obs, _ = env.reset()
    done, steps, total_volume_seen = False, 0, 0
    horizon = env._window_hi - env.current_step
    while not done:
        row = env.df.iloc[env.current_step]
        bid_vol = sum(float(row.get(f"bid_size_{i}", 0)) for i in range(1, 6))
        ask_vol = sum(float(row.get(f"ask_size_{i}", 0)) for i in range(1, 6))
        total_vol = bid_vol + ask_vol
        total_volume_seen += total_vol
        avg_vol = total_volume_seen / max(steps + 1, 1)
        if total_vol > avg_vol * 1.2 and steps < horizon - 100:
            action = 1
        elif steps >= horizon - 50 and env.inventory > 0:
            action = 2
        elif total_vol < avg_vol * 0.8 and env.inventory > 0:
            action = 2
        else:
            action = 0
        obs, r, done, trunc, info = env.step(action)
        steps += 1
        if trunc:
            break
    stats = env.get_episode_stats(); stats["name"] = "VWAP"
    return stats


def run_random(env, seed=42):
    obs, _ = env.reset()
    done, rng = False, np.random.RandomState(seed)
    while not done:
        obs, r, done, trunc, info = env.step(rng.randint(0, 5))
        if trunc:
            break
    stats = env.get_episode_stats(); stats["name"] = "Random"
    return stats

In [ ]:
all_stats = []
all_stats.append(evaluate_agent(cvml_model, make_env(train=False), "PPO+CVML+Macro"))
all_stats.append(evaluate_agent(nomacro_model, make_env(train=False, zero_macro=True), "PPO+CVML"))
all_stats.append(evaluate_agent(flat_model, make_env(train=False), "PPO+FlatMLP"))
for runner in (run_buy_hold, run_twap, run_vwap, run_random):
    all_stats.append(runner(make_env(train=False)))

headers = ["Model", "Return%", "Sharpe", "MaxDD%", "Trades", "Buy", "Sell", "Final PV", "Inv"]
widths = [18, 10, 10, 10, 8, 6, 6, 12, 8]
log_line("\n" + "=" * 98)
log_line("OUT-OF-SAMPLE EVALUATION (held-out last 30% of data)".center(98))
log_line("=" * 98)
log_line("".join(f"{h:>{w}}" for h, w in zip(headers, widths)))
log_line("-" * 98)
for s in all_stats:
    log_line(f"{s['name']:>18}{s['total_return_pct']:>10.4f}{s['sharpe_ratio']:>10.4f}"
             f"{s['max_drawdown_pct']:>10.4f}{s['total_trades']:>8}{s['buy_trades']:>6}"
             f"{s['sell_trades']:>6}  ${s['final_pv']:>9.2f}{s['final_inventory']:>8}")
log_line("=" * 98)

## 6. Equity Curves (test window)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
for model, label in ((cvml_model, "PPO+CVML+Macro"),
                     (nomacro_model, "PPO+CVML (no macro)"),
                     (flat_model, "PPO+FlatMLP")):
    env = make_env(train=False, zero_macro=(label == "PPO+CVML (no macro)"))
    evaluate_agent(model, env, label)
    ax.plot(env.portfolio_history, label=label, linewidth=1.2)

env = make_env(train=False)
run_buy_hold(env)
ax.plot(env.portfolio_history, label="Buy&Hold", linestyle="--", linewidth=1.2)

ax.set_xlabel("Test-window step"); ax.set_ylabel("Portfolio value ($)")
ax.set_title("Out-of-sample equity curves")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "equity_curves.png"), dpi=120)
plt.show()
log_line(f"\nSaved equity_curves.png; artifacts in {OUT_DIR}")

## 🚀 Scale up
This was a **10,000-step smoke test**. For a real run, edit
`TRAIN_TIMESTEPS = 500_000` in `build_notebook.py`, regenerate, and push again.
Macro vectors: `python macro/run_swarm_batch.py --max-events 500` locally, then
`./kaggle_sync.sh data-update` so the dataset carries the refreshed `macro_vectors.npz`.